# Mutual Information, and What Cleaning Did to the Answers

> **Notebook 6 of 6. This one is optional.** The four before it are the core path for the
> day and they stand on their own. Do this one if you finish them with time to spare, or
> come back to it before the Friday project, where the question it answers comes up for
> real.

Every measure in `05_correlation.ipynb` assumes something about the
*shape* of a relationship. Pearson assumes a straight line, Spearman and Kendall assume a
consistent direction. When the real relationship has neither, all of them report
approximately nothing, and a careless analyst concludes there is nothing there.

This notebook covers the measure that assumes nothing, and then closes the loop: it
measures what the imputation choices you made in `03_missing_data.ipynb` did to the
correlations you have just been computing.

First the libraries this notebook needs.

In [ ]:
# data analysis
import numpy as np
import pandas as pd

# visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# the production mutual information estimator
from sklearn.feature_selection import mutual_info_regression

sns.set_style("whitegrid")

Then the data, as the previous notebook left it.

In [ ]:
# the cleaned data plus the month and season columns derived in 04_correlation
df_weather = pd.read_csv("../data/seattle-weather_features.csv", parse_dates=["date"])
df_weather.head()

## 1. When correlation lies: mutual information

Every measure so far assumes something about the *shape* of a relationship. Pearson assumes
a straight line. Spearman assumes a consistent direction. When the real relationship has
neither, both report approximately nothing, and a careless analyst concludes there is
nothing there.

This dataset contains a perfect example. You have a `month` column. Temperature obviously
depends on the month. Ask Pearson.

In [ ]:
# Pearson and Spearman between month and temp_max
print(f"Pearson  month vs temp_max : {df_weather.month.corr(df_weather.temp_max):.3f}")
print(f"Spearman month vs temp_max : {df_weather.month.corr(df_weather.temp_max, method='spearman'):.3f}")

Both close to `0.1`. On that evidence you would throw the month away. Now look at it.

In [ ]:
# the relationship those two numbers missed
plt.figure(figsize=(6, 3.5), dpi=100)
sns.lineplot(x=df_weather.month, y=df_weather.temp_max, marker="o", errorbar=None)
plt.xticks(range(1, 13))
plt.title("mean temp_max by month");

The relationship is enormous, running from about `8` degrees in January to `26` in July and
back down again. Pearson misses it because it is **not monotonic**: temperature rises for
half the year and falls for the other half, and the two halves cancel out.

**Mutual information** has no shape assumption. It comes from information theory and
measures how much knowing one variable reduces your uncertainty about the other:

$$
I(X;Y) = \sum_{x}\sum_{y} p(x,y) \, \log \frac{p(x,y)}{p(x)\,p(y)}
$$

If `X` and `Y` are independent then `p(x,y) = p(x)p(y)`, the logarithm is `log(1) = 0`, and
the whole sum is `0`. Any departure from independence, of any shape, makes it positive.

Two practical differences from correlation:

- **it has no upper bound of 1** and no sign. You compare values against each other, not
  against a fixed scale.
- **it needs a probability table**, so continuous columns have to be **binned** first. The
  number of bins is a choice, and it affects the answer.

In [ ]:
def mutual_information(column_a, column_b, bins=10):
    """Mutual information between two Series, in nats.

    Numerical columns are binned into equal-width intervals first, because the estimate
    is computed from a contingency table of counts.

    Args:
        column_a: any pandas Series.
        column_b: any pandas Series of the same length.
        bins: how many intervals to cut numerical columns into.

    Returns:
        A float >= 0. 0 means the two columns are independent in this table.
    """
    def discretise(column):
        return pd.cut(column, bins) if pd.api.types.is_numeric_dtype(column) else column

    crosstab = pd.crosstab(discretise(column_a), discretise(column_b))
    counts = crosstab.to_numpy()

    joint = counts / counts.sum()                     # p(x, y)
    p_a = joint.sum(axis=1, keepdims=True)            # p(x)
    p_b = joint.sum(axis=0, keepdims=True)            # p(y)

    # cells with zero probability contribute nothing, and log(0) is undefined
    non_zero = joint > 0
    independent = (p_a @ p_b)[non_zero]
    return float((joint[non_zero] * np.log(joint[non_zero] / independent)).sum())

> **Exercise:**
>
> Compute the mutual information between `temp_max` and each of `month`, `temp_min` and
> `wind`. Treat `month` as a categorical label by converting it with `.astype(str)`, since
> its numeric value carries no meaning beyond identifying the month.
>
> Then answer in a comment: Pearson said `month` was worth about `0.13` and `temp_min`
> about `0.88`. What does mutual information say, and which of the two measures would have
> led you to a better model?

<details><summary>
Click here for a hint...
</summary>

Drop the rows where `temp_max` is missing first, with
`df_weather.dropna(subset=["temp_max"])`, so all three comparisons use the same rows.
</details>

In [ ]:
# use the same rows for all three, so the numbers are comparable
complete = df_weather.dropna(subset=["temp_max"])

print(f"temp_max vs month    : {mutual_information(complete.month.astype(str), complete.temp_max):.3f}")
print(f"temp_max vs temp_min : {mutual_information(complete.temp_min, complete.temp_max):.3f}")
print(f"temp_max vs wind     : {mutual_information(complete.wind, complete.temp_max):.3f}")

# Mutual information puts month at about 0.68 and temp_min at about 0.74, so month is
# nearly as informative about the maximum temperature as the minimum temperature is.
# Pearson put month at 0.13 and temp_min at 0.88, which would have led us to drop the
# month entirely. Mutual information is the measure that would have produced the better
# model here, because the relationship is real but not linear.

The version above is written out longhand because the formula is the lesson. In practice
you would reach for **`mutual_info_regression`** or **`mutual_info_classif`** from
`sklearn.feature_selection`, which estimate mutual information from nearest-neighbour
distances rather than from bins, so there is no bin count to choose.

It is worth checking one against the other.

In [ ]:
# the production estimator, for comparison with the longhand version above.
# sklearn rejects missing values, so drop any row with a gap in the columns used here
complete_rows = df_weather[["month", "temp_min", "wind", "precipitation", "temp_max"]].dropna()
features = complete_rows[["month", "temp_min", "wind", "precipitation"]]

scores = mutual_info_regression(
    features,
    complete_rows.temp_max,
    discrete_features=[True, False, False, False],  # month is a label, the rest continuous
    random_state=42,                                # the estimator is stochastic
)

for name, score in zip(features.columns, scores):
    print(f"temp_max vs {name:14}: {score:.3f}")

Both versions agree on the ordering that matters: `month` sits close behind `temp_min`, and
far ahead of `wind` and `precipitation`. Rank the same four features by the size of their
Pearson correlation with `temp_max` and `month` comes **last**. That is the whole lesson of
this section in one comparison.

The exact values differ between the two estimators, which is a useful reminder that mutual
information is **estimated**, not computed. Neither number is the truth, and a gap of
`0.05` between two features is not a finding.

So why not use mutual information for everything? Because the shape it ignores is often the
shape you want. Pearson tells you the *direction* of a relationship and mutual information
does not, it needs more data to estimate reliably, and its value has no natural
interpretation. Use correlation first, and reach for mutual information when correlation
reports nothing and you do not believe it.

## 2. What your cleaning choices did to the answers

Cleaning and exploring are not separate activities. Every decision you made in
`03_missing_data.ipynb` has changed the numbers you have just computed, and this section
makes that visible.

`temp_max` is missing on about a quarter of all rows. In `03_missing_data.ipynb` you tried
three ways of dealing with that: dropping the rows, filling with the mean, and interpolating. Each
produces a different `temp_max` column, so each produces a different correlation with
`temp_min`.

Below are the three treatments side by side, using exactly the techniques you applied in
the Imputing section of `03_missing_data.ipynb`. Read the code as a summary of what you
have already done,
then look hard at the table it prints.

In [ ]:
# build the three versions
dropped = df_weather.dropna(subset=["temp_max"]).copy()

mean_filled = df_weather.copy()
mean_filled["temp_max"] = mean_filled.temp_max.fillna(mean_filled.temp_max.mean())

interpolated = df_weather.copy()
interpolated["temp_max"] = interpolated.temp_max.interpolate()

# compare correlation and standard deviation across them
comparison = pd.DataFrame(
    [
        {
            "treatment": name,
            "pearson_r": frame.temp_max.corr(frame.temp_min),
            "temp_max_std": frame.temp_max.std(),
        }
        for name, frame in [
            ("dropped", dropped),
            ("mean_filled", mean_filled),
            ("interpolated", interpolated),
        ]
    ]
)
comparison.round(3)

Three treatments of the same column, three different answers:

| treatment | Pearson r | std of temp_max |
|---|---|---|
| dropped | 0.880 | 7.43 |
| mean filled | 0.777 | 6.48 |
| interpolated | 0.880 | 7.31 |

**Mean imputation shrinks the spread and pulls the correlation towards zero.** That is not
bad luck, it is arithmetic: you have inserted 349 copies of the mean, which are by
definition at zero distance from it, so the variance must fall. And because those inserted
values carry no relationship to `temp_min` at all, they dilute the relationship that the
real rows do have.

Interpolation avoids both problems here because the data is a time series: the temperature
on a missing day really is close to the temperature on the days either side.

Two things to take away:

1. **There is no neutral way to handle missing data.** Doing nothing is also a choice, and
   dropping a quarter of your rows has consequences of its own.
2. **Imputation is part of your analysis, not preparation for it.** Report what you did.
   A correlation of `0.78` and one of `0.88` support quite different conclusions, and the
   only thing separating them here is a line of code.

> **Warning.** There is a further trap once you start building models. If you compute the mean from the
> whole dataset and then split into training and test sets, the test set has influenced the
> filled values, and your evaluation is optimistic. This is called **data leakage**, and it
> matters as soon as you split data to train and evaluate a model.

## Wrapping up

Two things this notebook added to the day.

**A measure that assumes nothing.** Mutual information found a relationship between `month`
and `temp_max` that Pearson, Spearman and Kendall all scored near zero, because the
relationship rises for half the year and falls for the other half. Ranked by correlation,
`month` was the least useful of four features; ranked by mutual information it was the
second most useful. The lesson is not that correlation is broken, it is that **a
coefficient of zero means no relationship of that shape**, and you only find out which
shape by drawing the picture.

**Your cleaning was part of your analysis.** Three defensible ways of filling one column
gave correlations of `0.880`, `0.777` and `0.880`. Nothing about the data changed, only a
line of code in `03_missing_data.ipynb`. Report what you did to the missing values, because
a reader cannot reconstruct it from your results.

Both come up for real on Friday. The project dataset has missing values, it has categorical
and numerical columns that need relating to each other, and it has a target you will be
asked to find useful features for.